In [ ]:
import os
DREAMS_DATA = os.environ.get("DREAMS_DATA", "/path/to/DREAMS")
NEHOD_TORCH = os.environ.get("NEHOD_TORCH", "/path/to/nehod_torch")

import numpy as np
from scipy.stats import binned_statistic as bs
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import h5py
import DREAMS

In [ ]:
def get_fof_tree(tree_cat, mw_idx, start_snap=90):

    nnodes = len(tree_cat['SubhaloID'])

    target_cut = np.in1d(tree_cat['SubhaloGrNr'], mw_idx) & np.in1d(tree_cat['SnapNum'], start_snap)
    target  = tree_cat['FirstSubhaloInFOFGroupID'][target_cut][0]
    target_idx = tree_cat['SubhaloID'] == target

    start = tree_cat['MainLeafProgenitorID'][target_idx]
    start_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == start][0]
    start_snap = tree_cat['SnapNum'][start_idx]

    fof_list = {start_snap:start_idx}
    next_ID = tree_cat['DescendantID'][start_idx]
    next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == next_ID][0]
    while next_ID != -1: ## While a descendant exists
        this_snap           = tree_cat['SnapNum'][next_idx]           ## Get my current snapshot
        fof_list[this_snap] = next_idx       ## Get the FOF group of the progenitor
        next_ID = tree_cat['DescendantID'][next_idx]

        if next_ID == -1:
            break

        next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == next_ID][0]

    return fof_list

def get_merger_tree(tree_cat, mw_idx, sub_idx, start_snap=90):

    nnodes = len(tree_cat['SubhaloID'])

    target_cut = (tree_cat['SubhaloGrNr'] == mw_idx) & (tree_cat['SnapNum'] == start_snap) & (tree_cat['SubfindID'] == sub_idx)
    target = tree_cat['SubhaloID'][target_cut][0]
    target_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == target][0]

    start = tree_cat['MainLeafProgenitorID'][target_idx]
    start_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == start][0]
    start_snap = tree_cat['SnapNum'][start_idx]

    fof_list = {start_snap:start_idx}
    next_ID = tree_cat['DescendantID'][start_idx]
    next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == next_ID][0]
    while next_ID != -1: ## While a descendant exists
        this_snap           = tree_cat['SnapNum'][next_idx]           ## Get my current snapshot
        fof_list[this_snap] = next_idx       ## Get the FOF group of the progenitor
        next_ID = tree_cat['DescendantID'][next_idx]

        if next_ID == -1:
            break

        next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == next_ID][0]

    return fof_list

def track_halo_center(tcat, fof_list):

    group_path = DREAMS_DATA + '/FOF_Subfind/CDM/MW_zooms/SB5/'

    scale_factors = []
    coordinates = []
    for snap in fof_list:
        scalef = DREAMS.get_scf(group_path, snap)
        pos = tcat['GroupPos'][fof_list[snap]]
        scale_factors.append(scalef)
        coordinates.append(pos)

    return np.array(scale_factors), np.array(coordinates)

def select_halo_stars(pcat, tree_cat, mw_idx):

    fof_list = get_fof_tree(tree_cat, mw_idx)
    halo_times, halo_coords = track_halo_center(tree_cat, fof_list)

    ftime = pcat['PartType4/GFM_StellarFormationTime']
    bpos = pcat['PartType4/BirthPos']

    wind_cut = ftime > 0
    ftime = ftime[wind_cut]
    bpos = bpos[wind_cut]

    xspl = CubicSpline(halo_times, halo_coords[:,0])
    yspl = CubicSpline(halo_times, halo_coords[:,1])
    zspl = CubicSpline(halo_times, halo_coords[:,2])
    halo_xpos = xspl(ftime)
    halo_ypos = yspl(ftime)
    halo_zpos = zspl(ftime)
    halo_pos = np.array([halo_xpos, halo_ypos, halo_zpos]).T

    halo_dist = np.sqrt(np.sum(np.square(halo_pos - bpos), axis=1)) / h
    halo_cut = halo_dist > 50

    return halo_cut

def select_merger_stars(pcat, tree_cat, mw_idx, sub_idx=-1, start_snap=90, radius=50):

    raw_fof_list = get_merger_tree(tree_cat, mw_idx, sub_idx, start_snap)
    fof_list = dict()
    for key in raw_fof_list:
        if key <= start_snap:
            fof_list[key] = raw_fof_list[key]
    halo_times, halo_coords = track_halo_center(tree_cat, fof_list)

    if len(halo_times) == 1:
        halo_times = np.array([halo_times[0], halo_times[0]+.01])
        halo_coords = np.array([halo_coords[0], halo_coords[0]+np.ones(3)*0.01])
    
    ftime = pcat['PartType4/GFM_StellarFormationTime']
    wind_cut = ftime > 0
    
    xspl = CubicSpline(halo_times, halo_coords[:,0])
    yspl = CubicSpline(halo_times, halo_coords[:,1])
    zspl = CubicSpline(halo_times, halo_coords[:,2])
    halo_xpos = xspl(ftime)
    halo_ypos = yspl(ftime)
    halo_zpos = zspl(ftime)
    halo_pos = np.array([halo_xpos, halo_ypos, halo_zpos]).T
    
    bpos = pcat['PartType4/BirthPos']
    halo_dist = np.sqrt(np.sum(np.square(halo_pos - bpos), axis=1)) / h
    
    halo_cut = (halo_dist < radius) & wind_cut

    return halo_cut

def get_halo_ids(galaxy_pcat, galaxy_cat, tree_cat, mw_idx):
    ftime = galaxy_pcat['PartType4/GFM_StellarFormationTime']
    wind_cut = ftime > 0
    halo_cut = select_halo_stars(galaxy_pcat, tree_cat, mw_idx)
    coords = galaxy_pcat['PartType4/Coordinates']
    r = np.sqrt(np.sum(np.square(coords[wind_cut] - galaxy_cat['GroupPos']),axis=1)) / h
    rcut = (r < 40) & (r > 5)
    central_ids = galaxy_pcat['PartType4/ParticleIDs'][wind_cut][halo_cut & rcut]
    return central_ids

In [ ]:
def cartesian_to_spherical_coords(x, y, z):
    r = np.sqrt(x**2 + y**2 + z**2)
    theta = np.arccos(z / r)  # polar angle
    phi = np.arctan2(y, x)    # azimuthal angle
    return r, theta, phi
    
def cartesian_velocities_to_spherical(x, y, z, vx, vy, vz):
    r, theta, phi = cartesian_to_spherical_coords(x, y, z)

    # Spherical unit vectors in Cartesian coordinates
    sin_theta = np.sin(theta)
    cos_theta = np.cos(theta)
    sin_phi = np.sin(phi)
    cos_phi = np.cos(phi)

    # Compute components
    v_r = (vx * sin_theta * cos_phi + 
           vy * sin_theta * sin_phi + 
           vz * cos_theta)

    v_theta = (vx * cos_theta * cos_phi + 
               vy * cos_theta * sin_phi - 
               vz * sin_theta)

    v_phi = (-vx * sin_phi + vy * cos_phi)

    return v_r, v_theta, v_phi

In [ ]:
def get_last_major_merger(tree_cat, mw_idx):

    nnodes = len(tree_cat['SubhaloID'])

    #initialize the search by finding the target halo in the tree
    target_cut = np.in1d(tree_cat['SubhaloGrNr'], mw_idx) & np.in1d(tree_cat['SnapNum'], 90)
    target  = tree_cat['FirstSubhaloInFOFGroupID'][target_cut][0]
    target_idx = tree_cat['SubhaloID'] == target

    start = tree_cat['MainLeafProgenitorID'][target_idx]
    start_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == start][0]
    start_snap = tree_cat['SnapNum'][start_idx]

    next_ID = tree_cat['DescendantID'][start_idx]
    next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == next_ID][0]
    
    #first get the mass of the central for every snapshot
    mass_over_time = dict()
    while next_ID != -1:
        this_snap = tree_cat['SnapNum'][next_idx]
        this_mass = tree_cat['SubhaloMassType'][next_idx,4]*1e10/h
        if this_mass > 0:
            mass_over_time[this_snap] = this_mass
        
        next_ID = tree_cat['DescendantID'][next_idx]
        if next_ID == -1:
            break
        next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == next_ID][0]

    next_ID = tree_cat['DescendantID'][start_idx]
    next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == next_ID][0]

    last_snap = []
    last_mass = []
    last_ratio = []
    last_fof = []
    last_sub = []
    while next_ID != -1: # While a descendant exists
        other_sat_id = tree_cat['NextProgenitorID'][next_idx]
        main_mass = tree_cat['SubhaloMassType'][next_idx,4]*1e10/h
        
        while other_sat_id > -1 and main_mass > 0: #search all halos that merge during this snapshot
            other_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == other_sat_id][0]
            other_orig = np.copy(other_idx)
            other_start = tree_cat['MainLeafProgenitorID'][other_idx]

            max_mass = 0
            max_snap = 0
            if other_start > other_sat_id: #continue if the progenitor has any history
                other_start_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == other_start][0]
                other_next_ID = tree_cat['DescendantID'][other_start_idx]
                other_next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == other_next_ID][0]
                while other_next_ID != other_sat_id: #get the maximum mass of the progenitor
                    other_mass = tree_cat['SubhaloMassType'][other_next_idx,4]*1e10/h
                    other_snap = tree_cat['SnapNum'][other_next_idx]
                    other_fof = tree_cat['SubhaloGrNr'][other_next_idx]
                    other_sub = tree_cat['SubfindID'][other_next_idx]
                    if other_mass > max_mass and other_snap in mass_over_time.keys(): #if there is a major merger, save it
                        max_mass = other_mass
                        max_snap = other_snap
                        max_fof = other_fof
                        max_sub = other_sub
                    other_next_ID = tree_cat['DescendantID'][other_next_idx]
                    if other_next_ID == -1:
                        break
                    other_next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == other_next_ID][0]

            #do the current snapshot
            other_mass = tree_cat['SubhaloMassType'][other_orig,4]*1e10/h
            this_snap = tree_cat['SnapNum'][other_orig]
            this_fof = tree_cat['SubhaloGrNr'][other_orig]
            this_sub = tree_cat['SubfindID'][other_orig]
            
            if other_mass > max_mass:
                max_mass = other_mass
                max_snap = this_snap
                max_fof = this_fof
                max_sub = this_sub

            #save merger if it happens while the MW exists
            if max_snap in mass_over_time:
                ratio = max_mass / mass_over_time[max_snap]
                if ratio > 1:
                    ratio = 1/ratio

                if ratio > 0.02: #limit to consider major merger
                    last_snap.append(max_snap)
                    last_mass.append(max_mass)
                    last_ratio.append(ratio)
                    last_fof.append(max_fof)
                    last_sub.append(max_sub)

            other_sat_id = tree_cat['NextProgenitorID'][other_idx]
        
        next_ID = tree_cat['DescendantID'][next_idx]
        if next_ID == -1:
            break
        next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == next_ID][0]

    #return the most recent merger
    if len(last_snap) > 0:
        idx = np.argmax(last_snap)
        latest_snap = last_snap[idx]
        latest_mass = last_mass[idx]
        latest_ratio = 1/last_ratio[idx]
        latest_fof = last_fof[idx]
        latest_sub = last_sub[idx]
    else:
        latest_snap = 1
        latest_mass = 1e5
        latest_ratio = 50
        latest_fof = None
        latest_sub = None
    
    return latest_snap, latest_mass, latest_ratio, latest_fof, latest_sub

In [ ]:
snap_path = DREAMS_DATA + '/Sims/CDM/MW_zooms/SB5/'
group_path = DREAMS_DATA + '/FOF_Subfind/CDM/MW_zooms/SB5/'
param_path = DREAMS_DATA + '/Parameters/CDM/MW_zooms/CDM_TNG_MW_SB5.txt'

In [ ]:
boxes = range(1024)        #which simulations you include, there are 1024 simulations
snap = 90                  #which redshift you use, right now the script only works for snap >~ 60
h = 0.6909                 #reduced hubble factor, don't change this
boxsize = 100000 / h 
G = 4.3e-6                 #kpc/Msun * (km/s)^2

In [ ]:
params = DREAMS.get_params_cdm(param_path)
params = np.array(params)

In [ ]:
keys = ['GroupMassType', 'GroupLenType', 'GroupPos', 'SubhaloGrNr', 'SubhaloPos', 'SubhaloMassType', 'SubhaloLenType', 'GroupFirstSub', 'GroupNsubs']
keys += ['SubhaloMass', 'SubhaloPos', 'GroupVel', 'GroupMass', 'Group_M_Crit200']
tree_keys = ['SubhaloGrNr','FirstSubhaloInFOFGroupID', 'SnapNum','SubfindID', 'SubhaloID', 'MainLeafProgenitorID', 'DescendantID', 'NextProgenitorID', 'FirstProgenitorID', 'SubhaloMassType', 'GroupPos', 'GroupMass', 'SubhaloPos']
part_keys = ['Coordinates', 'Masses', 'GFM_StellarFormationTime', 'BirthPos', 'ParticleIDs', 'Velocities']

In [ ]:
save_snaps = [0,2,6,11,14,18,24,32,44,52,61,67,73,81,90]
save_snaps = np.array(save_snaps)

In [ ]:
def get_all_major_merger(box, tree_cat, galaxy_pcat, galaxy_cat, mw_idx):

    nnodes = len(tree_cat['SubhaloID'])

    #initialize the search by finding the target halo in the tree
    target_cut = np.in1d(tree_cat['SubhaloGrNr'], mw_idx) & np.in1d(tree_cat['SnapNum'], 90)
    target  = tree_cat['FirstSubhaloInFOFGroupID'][target_cut][0]
    
    mass_over_time = dict()
    bad_times = []
    curr_ID = target
    
    # Walk backward in time along the main progenitor branch
    while curr_ID != -1:
        # Using np.where is generally faster than boolean indexing with np.arange
        curr_idx = np.where(tree_cat['SubhaloID'] == curr_ID)[0][0]
        
        this_snap = tree_cat['SnapNum'][curr_idx]
        this_mass = tree_cat['SubhaloMassType'][curr_idx, 1] * 1e10 / h
        mass_over_time[this_snap] = this_mass
        
        # Move to the primary progenitor in the previous snapshot
        curr_ID = tree_cat['FirstProgenitorID'][curr_idx]

    start = tree_cat['MainLeafProgenitorID'][target_cut][0]
    start_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == start][0]
    next_ID = tree_cat['DescendantID'][start_idx]
    next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == next_ID][0]

    last_snap = []
    last_mass = []
    last_ratio = []
    last_fof = []
    last_sub = []
    while next_ID != -1: # While a descendant exists
        other_sat_id = tree_cat['NextProgenitorID'][next_idx]
        main_mass = tree_cat['SubhaloMassType'][next_idx,1]*1e10/h
        
        while other_sat_id > -1 and main_mass > 0: #search all halos that merge during this snapshot
            other_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == other_sat_id][0]
            other_orig = np.copy(other_idx)
            other_start = tree_cat['MainLeafProgenitorID'][other_idx]

            max_mass = 0
            max_snap = 0
            if other_start > other_sat_id: #continue if the progenitor has any history
                other_start_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == other_start][0]
                other_next_ID = tree_cat['DescendantID'][other_start_idx]
                other_next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == other_next_ID][0]
                while other_next_ID != other_sat_id: #get the maximum mass of the progenitor
                    other_mass = tree_cat['SubhaloMassType'][other_next_idx,1]*1e10/h
                    other_snap = tree_cat['SnapNum'][other_next_idx]
                    other_fof = tree_cat['SubhaloGrNr'][other_next_idx]
                    other_sub = tree_cat['SubfindID'][other_next_idx]
                    if other_mass > max_mass and other_snap in mass_over_time.keys(): #if there is a major merger, save it
                        max_mass = other_mass
                        max_snap = other_snap
                        max_fof = other_fof
                        max_sub = other_sub
                    other_next_ID = tree_cat['DescendantID'][other_next_idx]
                    if other_next_ID == -1:
                        break
                    other_next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == other_next_ID][0]

            #do the current snapshot
            other_mass = tree_cat['SubhaloMassType'][other_orig,1]*1e10/h
            this_snap = tree_cat['SnapNum'][other_orig]
            this_fof = tree_cat['SubhaloGrNr'][other_orig]
            this_sub = tree_cat['SubfindID'][other_orig]
            
            if other_mass > max_mass:
                max_mass = other_mass
                max_snap = this_snap
                max_fof = this_fof
                max_sub = this_sub

            #save merger if it happens while the MW exists
            if max_snap in mass_over_time:
                ratio = max_mass / mass_over_time[max_snap]
                if ratio > 1:
                    ratio = 1/ratio

                if ratio > 0.05: #limit to consider major merger
                    last_snap.append(max_snap)
                    last_mass.append(max_mass)
                    last_ratio.append(ratio)
                    last_fof.append(max_fof)
                    last_sub.append(max_sub)

            other_sat_id = tree_cat['NextProgenitorID'][other_idx]
        
        next_ID = tree_cat['DescendantID'][next_idx]
        if next_ID == -1:
            break
        next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == next_ID][0]

    times = np.array(last_snap)
    cut = times > 10
    if np.sum(cut) > 3:
        return
    print("Box", box)
    print("Found", np.sum(cut), "Mergers")
    print(np.array(last_snap)[cut])
    print(np.array(last_ratio)[cut])
    print()
    return
    
    #search for radial mergers in all major mergers
    halo_ids = get_halo_ids(galaxy_pcat, galaxy_cat, tree_cat, mw_idx)

    all_f = []
    all_coords = []
    for i in range(len(last_snap)):

        merger_fof = last_fof[i]
        merger_sub = last_sub[i]

        merger_tree = get_merger_tree(tree_cat, merger_fof, merger_sub, last_snap[i])
        merger_snap = save_snaps[np.argmin(np.abs(last_snap[i] - save_snaps[save_snaps <= last_snap[i]]))]

        if merger_snap not in merger_tree:
            all_f.append(0)
            all_coords.append([[0],[0],[0]])
            continue
        
        merger_sub = tree_cat['SubfindID'][merger_tree[merger_snap]]
        
        gpath = f'{group_path}/box_{box}/fof_subhalo_tab_{merger_snap:03}.hdf5'
        spath = f'{snap_path}/box_{box}/snap_{merger_snap:03}.hdf5'
        mcat = DREAMS.load_group_data(gpath, keys)     #read DREAMS data
        mpcat = DREAMS.load_particle_data(spath, part_keys, part_types=[4])
        merger_pcat, merger_cat = DREAMS.get_galaxy_data(mpcat, mcat, -1, merger_sub)
        merger_ids = merger_pcat['PartType1/ParticleIDs']
    
        all_ids = galaxy_pcat['PartType4/ParticleIDs']
        halo_cut = np.in1d(all_ids, halo_ids)
        merger_cut = np.in1d(halo_ids, merger_ids)
        halo_mass = np.sum(galaxy_pcat['PartType4/Masses'][halo_cut]*1e10/h)
        merger_mass = np.sum(galaxy_pcat['PartType4/Masses'][halo_cut][merger_cut]*1e10/h)
        
        f = merger_mass / halo_mass

        if np.sum(merger_cut) > 0:
            coords = (galaxy_pcat['PartType4/Coordinates'][halo_cut][merger_cut]-galaxy_cat['GroupPos'])/h
            vels = galaxy_pcat['PartType4/Velocities'][halo_cut][merger_cut]-galaxy_cat['GroupVel']
            vr, vt, vp = cartesian_velocities_to_spherical(coords[:,0], coords[:,1], coords[:,2], vels[:,0], vels[:,1], vels[:,2])
            all_coords.append([vr, vt, vp])
        else:
            all_coords.append([[0],[0],[0]])

        all_f.append(f)

    all_f = np.array(all_f)
    fcut = all_f > 0.1 #only look at mergers that make up 10% of halo
    idx = np.arange(len(last_snap))[fcut]
    
    if len(idx) == 0: #if no mergers are 10% of halo take largest component
        idx = np.array([np.argmax(all_f)])

    all_vr = []
    all_vt = []
    all_vp = []
    for i in idx:
        all_vr.extend(list(all_coords[i][0]))
        all_vt.extend(list(all_coords[i][1]))
        all_vp.extend(list(all_coords[i][2]))
        
    sp = np.std(all_vp); st = np.std(all_vt); sr = np.std(all_vr)
    beta = 1 - (sp*sp + st*st)/(2*sr*sr)

    if np.isnan(beta):
        beta = 0

    f = np.sum(all_f[fcut])
    merger_idx = np.argmax(all_f[idx])
    merger_time = np.array(last_snap)[idx][merger_idx]
    merger_ratio = np.array(last_ratio)[idx][merger_idx]

    index = np.argsort(all_f[idx])[::-1]
    print(all_f[idx][index])
    print(np.array(last_snap)[idx][index])
    print(np.array(last_mass)[idx][index])
    print(np.array(last_ratio)[idx][index])
    print(beta)
    print()
    
    return np.max(last_snap), merger_time, merger_ratio, len(idx), f, beta

In [ ]:
def mass_history(tree_cat, galaxy_pcat, galaxy_cat, mw_idx):

    nnodes = len(tree_cat['SubhaloID'])

    #initialize the search by finding the target halo in the tree
    target_cut = np.in1d(tree_cat['SubhaloGrNr'], mw_idx) & np.in1d(tree_cat['SnapNum'], 90)
    target  = tree_cat['FirstSubhaloInFOFGroupID'][target_cut][0]
    
    mass_over_time = dict()
    bad_times = []
    curr_ID = target
    
    # Walk backward in time along the main progenitor branch
    while curr_ID != -1:
        # Using np.where is generally faster than boolean indexing with np.arange
        curr_idx = np.where(tree_cat['SubhaloID'] == curr_ID)[0][0]
        
        this_snap = tree_cat['SnapNum'][curr_idx]
        this_mass = np.sum(tree_cat['SubhaloMassType'][curr_idx, :] * 1e10 / h)

        if this_snap < 90:
            mass_ratio = mass_over_time[np.min(list(mass_over_time.keys()))] / this_mass
            snap_diff = np.min(list(mass_over_time.keys())) - this_snap
            if mass_ratio > 3 + snap_diff:
                curr_ID = tree_cat['FirstProgenitorID'][curr_idx]
                bad_times.append(this_snap)
                continue
        this_mass = tree_cat['SubhaloMassType'][curr_idx, 4] * 1e10 / h
        mass_over_time[this_snap] = this_mass
        
        # Move to the primary progenitor in the previous snapshot
        curr_ID = tree_cat['FirstProgenitorID'][curr_idx]
        
    return mass_over_time, bad_times

In [ ]:
def pos_history(tree_cat, galaxy_pcat, galaxy_cat, mw_idx):

    nnodes = len(tree_cat['SubhaloID'])

    #initialize the search by finding the target halo in the tree
    target_cut = np.in1d(tree_cat['SubhaloGrNr'], mw_idx) & np.in1d(tree_cat['SnapNum'], 90)
    target  = tree_cat['FirstSubhaloInFOFGroupID'][target_cut][0]
    
    pos_over_time = dict()
    times = []
    fof_idx = []
    curr_ID = target
    
    # Walk backward in time along the main progenitor branch
    while curr_ID != -1:
        # Using np.where is generally faster than boolean indexing with np.arange
        curr_idx = np.where(tree_cat['SubhaloID'] == curr_ID)[0][0]
        
        this_snap = tree_cat['SnapNum'][curr_idx]
        this_fof = tree_cat['SubhaloGrNr'][curr_idx]
        #pos_over_time[this_snap] = this_pos
        times.append(this_snap)
        fof_idx.append(this_fof)
        
        # Move to the primary progenitor in the previous snapshot
        curr_ID = tree_cat['FirstProgenitorID'][curr_idx]

    return times, fof_idx

In [ ]:
mass = []
smass = []
sim_params = []
mm_time = []
gse_time = []
gse_ratio = []
ngse = []
halo_f = []
halo_beta = []
good_boxes = []
cnt = 0
# fig, ax = plt.subplots(1,3, figsize=(15,5))
fig, ax = plt.subplots()
# for box in range(1024):
for box in [1,56,102,145,189,192,197,217,250,255,260,263,269,296,303,311,318,320,333,338,341,355,366,393,406,441,459,473,534,566,621,676,692,771,782,818,827,842,872,907,960,971,989,998,1016,1020]:

    #if box == 698:
    #    continue
    
    snap = 90
    gpath = f'{group_path}/box_{box}/fof_subhalo_tab_{snap:03}.hdf5'
    spath = f'{snap_path}/box_{box}/snap_{snap:03}.hdf5'
    tpath = f'{group_path}/box_{box}/tree_extended.hdf5'

    # if len(mass) % 50 == 0:
    #    print(len(mass))
    
    cat = DREAMS.load_group_data(gpath, keys)     #read DREAMS data
    mw_idx = DREAMS.get_MW_idx(cat)        #find target MW
    sub_idx = cat['GroupFirstSub'][mw_idx]
    tree_cat = DREAMS.load_sublink_cat(tpath, tree_keys)

    if len(tree_cat['SubhaloGrNr']) != len(tree_cat['SnapNum']):
        print(box, "!!!")
        continue

    pcat = DREAMS.load_particle_data(spath, part_keys, part_types=[4])
    galaxy_pcat, galaxy_cat = DREAMS.get_galaxy_data(pcat, cat, mw_idx, 0)
    
    try:
        get_all_major_merger(box, tree_cat, galaxy_pcat, galaxy_cat, mw_idx)
    except:
        print("box", box, "failed")
    continue

    
    recent_snap, merger_snap, merger_ratio, nmerger, f, beta = get_all_major_merger(tree_cat, galaxy_pcat, galaxy_cat, mw_idx)
    print(recent_snap, merger_snap, merger_ratio, nmerger, f, beta)
    # cnt += recent_snap
    # print(box, cnt)

    # mass_over_time, bad_times = mass_history(tree_cat, galaxy_pcat, galaxy_cat, mw_idx)
    # if len(bad_times) <= 10:
    #     continue
    # p = ax.plot(mass_over_time.keys(), mass_over_time.values())
    # ax.vlines(bad_times,ymin=1e9, ymax=1e12, linestyle='--', alpha=0.5, color=p[0].get_color())
    # print(box, len(bad_times))

    # pos_over_time = pos_history(tree_cat, galaxy_pcat, galaxy_cat, mw_idx)
    # time = np.array(list(pos_over_time.keys()))
    # pos = np.array(list(pos_over_time.values()))
    # ax[0].plot(time, pos[:,0])
    # ax[1].plot(time, pos[:,1])
    # ax[2].plot(time, pos[:,2])

    # snaps, fof = pos_history(tree_cat, galaxy_pcat, galaxy_cat, mw_idx)
    # np.save(f"fof_arrays/box_{box}_snaps", snaps)
    # np.save(f"fof_arrays/box_{box}_fofs", fof)
    # continue

    recent_sf = DREAMS.get_scf(group_path, recent_snap)
    merger_sf = DREAMS.get_scf(group_path, merger_snap)

    mm_time.append(recent_sf)
    gse_time.append(merger_sf)
    gse_ratio.append(merger_ratio)
    ngse.append(nmerger)
    halo_f.append(f)
    halo_beta.append(beta)
    
    this_mass = cat['Group_M_Crit200'][mw_idx]*1e10/h
    this_smass = cat['SubhaloMassType'][sub_idx,4]*1e10/h
    mass.append(this_mass)
    smass.append(this_smass)
    
    sim_params.append(params[box])
    good_boxes.append(box)

plt.xlabel("Snapshot")
plt.ylabel("Total Mass [M$_\odot$]")
plt.yscale("log")
plt.show()
# fig.savefig("plots/mass_accumulation.pdf", bbox_inches='tight')

In [ ]:
mass = np.array(mass)
smass = np.array(smass)
sim_params = np.array(sim_params)
mm_time = np.array(mm_time)
gse_time = np.array(gse_time)
gse_ratio = 1/np.array(gse_ratio)
ngse = np.array(ngse)
halo_f = np.array(halo_f)
halo_beta = np.array(halo_beta)
good_boxes = np.array(good_boxes)

In [ ]:
om = sim_params[:,0]
s8 = sim_params[:,1]
sn1 = sim_params[:,2]
sn2 = sim_params[:,3]
agn1 = sim_params[:,4]

In [ ]:
data_cond = np.array([np.log10(mass),
                      np.log10(smass),
                      om,
                      s8,
                      np.log10(sn1), 
                      np.log10(sn2), 
                      np.log10(agn1),
                      mm_time,
                      gse_time,
                      gse_ratio,
                      ngse,
                      halo_f,
                      halo_beta
                      ]).T

In [ ]:
# header = 'total_mass,stellar_mass,omega_m,sigma_8,log_sn1,log_sn2,log_agn1,mm_time,gse_time,gse_ratio,ngse,halo_f,halo_beta'
# np.savetxt("nehod_data/gse_prob_dylan_cond.csv", data_cond, delimiter=',', header=header, comments='')
# np.save("nehod_data/gse_prob_dylan_boxes", good_boxes)

In [ ]:
stop

In [ ]:
fcut = halo_f > 0.5
bcut = halo_beta > 0.5
ncut = ngse < 3
gse_cut = fcut & bcut & ncut
print(np.sum(gse_cut))

tcut = gse_time < 0.5
mcut = gse_ratio < 10
gse_cut = tcut & mcut & fcut & bcut & ncut
print(np.sum(gse_cut))
print(np.sum(tcut), np.sum(mcut), np.sum(fcut), np.sum(bcut), np.sum(ncut))

In [ ]:
mass = []
smass = []
sim_params = []
mm_time = []
mm_mass = []
mm_ratio = []
halo_f = []
halo_beta = []
good_boxes = []
for box in boxes:

    if box == 698:
        continue
    
    snap = 90
    gpath = f'{group_path}/box_{box}/fof_subhalo_tab_{snap:03}.hdf5'
    spath = f'{snap_path}/box_{box}/snap_{snap:03}.hdf5'
    tpath = f'{group_path}/box_{box}/tree_extended.hdf5'

    if len(mass) % 100 == 0:
       print(len(mass))
    
    cat = DREAMS.load_group_data(gpath, keys)     #read DREAMS data
    mw_idx = DREAMS.get_MW_idx(cat)        #find target MW
    sub_idx = cat['GroupFirstSub'][mw_idx]
    tree_cat = DREAMS.load_sublink_cat(tpath, tree_keys)

    if len(tree_cat['SubhaloGrNr']) != len(tree_cat['SnapNum']):
        print(box, "!!!")
        continue
    
    snap, gse_mass, ratio, merger_fof, merger_sub = get_last_major_merger(tree_cat, mw_idx)
    scalef = DREAMS.get_scf(group_path, snap)

    pcat = DREAMS.load_particle_data(spath, part_keys, part_types=[4])
    galaxy_pcat, galaxy_cat = DREAMS.get_galaxy_data(pcat, cat, mw_idx, 0)
    halo_ids = get_halo_ids(galaxy_pcat, galaxy_cat, tree_cat, mw_idx)

    merger_tree = get_merger_tree(tree_cat, merger_fof, merger_sub, snap)
    merger_snap = save_snaps[np.argmin(np.abs(snap - save_snaps[save_snaps <= snap]))]

    if merger_snap in merger_tree and snap-merger_snap<5:
        gpath = f'{group_path}/box_{box}/fof_subhalo_tab_{merger_snap:03}.hdf5'
        spath = f'{snap_path}/box_{box}/snap_{merger_snap:03}.hdf5'
        mcat = DREAMS.load_group_data(gpath, keys)     #read DREAMS data
        mpcat = DREAMS.load_particle_data(spath, part_keys, part_types=[4])
        merger_pcat, merger_cat = DREAMS.get_galaxy_data(mpcat, mcat, merger_fof, merger_sub)
        merger_ids = merger_pcat['PartType4/ParticleIDs']
    else:
        merger_cut = select_merger_stars(galaxy_pcat, tree_cat, merger_fof, merger_sub, snap, radius=50)
        merger_ids = galaxy_pcat['PartType4/ParticleIDs'][merger_cut]
    
    all_ids = galaxy_pcat['PartType4/ParticleIDs']
    halo_cut = np.in1d(all_ids, halo_ids)
    merger_cut = np.in1d(halo_ids, merger_ids)
    halo_mass = np.sum(galaxy_pcat['PartType4/Masses'][halo_cut]*1e10/h)
    merger_mass = np.sum(galaxy_pcat['PartType4/Masses'][halo_cut][merger_cut]*1e10/h)

    f = merger_mass / halo_mass

    if np.sum(merger_cut) > 0:
        coords = galaxy_pcat['PartType4/Coordinates'][halo_cut][merger_cut]-galaxy_cat['GroupPos']
        vels = galaxy_pcat['PartType4/Velocities'][halo_cut][merger_cut]-galaxy_cat['GroupVel']
    else:
        coords = galaxy_pcat['PartType4/Coordinates'][halo_cut]-galaxy_cat['GroupPos']
        vels = galaxy_pcat['PartType4/Velocities'][halo_cut]-galaxy_cat['GroupVel']
    
    vr, vt, vp = cartesian_velocities_to_spherical(coords[:,0], coords[:,1], coords[:,2], vels[:,0], vels[:,1], vels[:,2])
    sp = np.std(vp); st = np.std(vt); sr = np.std(vr)
    beta = 1 - (sp*sp + st*st)/(2*sr*sr)
    
    halo_f.append(f)
    halo_beta.append(beta)
    
    mm_time.append(scalef)
    mm_ratio.append(ratio)
    mm_mass.append(gse_mass)

    this_mass = cat['Group_M_Crit200'][mw_idx]*1e10/h
    this_smass = cat['SubhaloMassType'][sub_idx,4]*1e10/h
    mass.append(this_mass)
    smass.append(this_smass)
    
    sim_params.append(params[box])
    good_boxes.append(box)

In [ ]:
mass = np.array(mass)
smass = np.array(smass)
sim_params = np.array(sim_params)
mm_time = np.array(mm_time)
mm_mass = np.array(mm_mass)
mm_ratio = np.array(mm_ratio)
halo_f = np.array(halo_f)
halo_beta = np.array(halo_beta)
good_boxes = np.array(good_boxes)

In [ ]:
om = sim_params[:,0]
s8 = sim_params[:,1]
sn1 = sim_params[:,2]
sn2 = sim_params[:,3]
agn1 = sim_params[:,4]

In [ ]:
data_cond = np.array([np.log10(mass),
                      np.log10(smass),
                      om,
                      s8,
                      np.log10(sn1), 
                      np.log10(sn2), 
                      np.log10(agn1),
                      mm_time,
                      np.log10(mm_mass),
                      mm_ratio,
                      halo_f,
                      halo_beta
                      ]).T

In [ ]:
beta_cut = np.isnan(halo_beta)
data_cond = data_cond[~beta_cut]
good_boxes = good_boxes[~beta_cut]

In [ ]:
header = 'total_mass,stellar_mass,omega_m,sigma_8,log_sn1,log_sn2,log_agn1,mm_time,mm_mass,mm_ratio,halo_f,halo_beta'
# Release: disabled -- this old 12-column export overwrote the GSE-emulator training file when the
# notebook was run top to bottom. The training file is built in the (commented) cell 58.
# np.savetxt("nehod_data/gse_prob_cond.csv", data_cond, delimiter=',', header=header, comments='')
# np.save("nehod_data/gse_prob_boxes", good_boxes)

In [ ]:
len(good_boxes)

In [ ]:
len(data_cond)

In [ ]:
[np.sum(np.isnan(el)) for el in data_cond.T]

In [ ]:
data = np.loadtxt("nehod_data/gse_prob_cond.csv",delimiter=',',skiprows=1)

In [ ]:
data[0]

In [ ]:
mm_time = data[:,7]
mm_ratio = data[:,9]
halo_f = data[:,10]
halo_beta = data[:,11]

In [ ]:
tcut = mm_time < 0.5
mcut = mm_ratio < 4
fcut = halo_f > 0.5
bcut = halo_beta > 0.5
cut = tcut & mcut & fcut & bcut
print(np.sum(cut))
print(np.sum(tcut), np.sum(mcut), np.sum(fcut), np.sum(bcut))

In [ ]:
props = ['halo_tot_smass','halo_sat_smass', 'halo_bmass', 'halo_bmass_frac', 'halo_sfr'] #galaxy
props += ['bulge_reff', 'bulge_mass', 'bulge_smass', 'bulge_bhmass', 'bulge_dm_frac', 'bulge_vdisp'] # bulge
props += ['disk_vcirc', 'disk_smass', 'disk_size', 'disk_height'] # disk
props += ['halo_central_smass', 'halo_break', 'halo_slope_in', 'halo_slope_out', 'halo_q_in', 'halo_q_out'] #halo

In [ ]:
data = dict()
boxes = dict()
for name in props:
    data[name] = np.load(f"violin_data/{name}.npy")
    boxes[name] = np.load(f"violin_data/weights/{name}.npy")

In [ ]:
all_boxes = []
for name in data:
    print(name)
    this_box = boxes[name]
    cut = np.isnan(data[name]) | np.isinf(data[name])
    all_boxes.append(list(this_box[~cut]))

In [ ]:
[el[0] for el in all_boxes]

In [ ]:
uboxes = []
for box in range(1024):
    is_in_box = True
    for i in range(len(all_boxes)):
        if box not in all_boxes[i]:
            is_in_box=False
    if is_in_box:
        uboxes.append(box)

In [ ]:
print([len(el) for el in all_boxes])
len(uboxes)

In [ ]:
clean_data = {name:[] for name in data}
clean_boxes = []
for box in range(1024):
    if box not in uboxes:
        continue
    clean_boxes.append(box)
    for name in data:
        cut = np.in1d(boxes[name], box)
        clean_data[name].append(data[name][cut][0])

In [ ]:
for name in clean_data:
    clean_data[name] = np.array(clean_data[name])
clean_boxes = np.array(clean_boxes)

In [ ]:
[(name,clean_data[name].shape) for name in clean_data]

In [ ]:
# gse_boxes = np.load("nehod_data/gse_prob_boxes.npy")
# gse_data = np.loadtxt("nehod_data/gse_prob_cond.csv", delimiter=',', skiprows=1)
gse_boxes = np.load("nehod_data/gse_prob_boxes_from_sims.npy")
gse_data = np.loadtxt("nehod_data/gse_prob_from_sims.csv", delimiter=',', skiprows=1)

In [ ]:
gse_data.shape

In [ ]:
gse_cut = np.in1d(gse_boxes, clean_boxes)
dreams_cut = np.in1d(clean_boxes, gse_boxes)
print(np.sum(gse_cut))

In [ ]:
clean_data.keys(), clean_data['halo_break'].shape

In [ ]:
for name1 in clean_data.keys():
    for name2 in clean_data.keys():
        plt.figure()
        plt.plot(clean_data[name1],clean_data[name2], '.')
        plt.ylabel(name2)
        plt.xlabel(name1)
        plt.show()
        break

In [ ]:
for name in clean_data:
    clean_data[name] = clean_data[name][dreams_cut]
gse_sort = np.argsort(gse_boxes[gse_cut])
gse_clean_data = gse_data[gse_cut][gse_sort]
print(gse_clean_data.shape)

In [ ]:
plt.plot(gse_clean_data[:,1], clean_data['halo_tot_smass'], '.')

In [ ]:
out = np.array(list(gse_clean_data.T) + [clean_data[name] for name in clean_data]).T

for name in clean_data.keys():
    print(name)

In [ ]:
out[0]

In [ ]:
out.shape

In [ ]:
# header = 'total_mass,stellar_mass,omega_m,sigma_8,log_sn1,log_sn2,log_agn1,mm_time,gse_time,gse_ratio,ngse,halo_f,halo_beta,halo_central_smass,halo_break,halo_slope_in,halo_slope_out,halo_q_in,halo_q_out'
# np.savetxt("nehod_data/gse_prob_halo_cond.csv", out, delimiter=',', header=header, comments='')

In [ ]:
# header = 'total_mass,stellar_mass,omega_m,sigma_8,log_sn1,log_sn2,log_agn1,mm_time,gse_time,gse_ratio,ngse,halo_f,halo_beta,halo_tot_smass,halo_sat_smass,halo_bmass,halo_bmass_frac,halo_sfr'
# np.savetxt("nehod_data/gse_prob_galaxy_cond.csv", out, delimiter=',', header=header, comments='')

In [ ]:
# header = 'total_mass,stellar_mass,omega_m,sigma_8,log_sn1,log_sn2,log_agn1,mm_time,gse_time,gse_ratio,ngse,halo_f,halo_beta,bulge_reff,bulge_mass,bulge_smass,bulge_bhmass,bulge_dm_frac,bulge_vdisp'
# np.savetxt("nehod_data/gse_prob_bulge_cond.csv", out, delimiter=',', header=header, comments='')

In [ ]:
# header = 'total_mass,stellar_mass,omega_m,sigma_8,log_sn1,log_sn2,log_agn1,mm_time,gse_time,gse_ratio,ngse,halo_f,halo_beta,disk_vcirc,disk_smass,disk_size,disk_height'
# np.savetxt("nehod_data/gse_prob_disk_cond.csv", out, delimiter=',', header=header, comments='')

In [ ]:
header = 'total_mass,stellar_mass,omega_m,sigma_8,log_sn1,log_sn2,log_agn1,mm_time,gse_time,gse_ratio,ngse,halo_f,halo_beta,'
header += 'halo_tot_smass,halo_sat_smass,halo_bmass,halo_bmass_frac,halo_sfr,'
header += 'bulge_reff,bulge_mass,bulge_smass,bulge_bhmass,bulge_dm_frac,bulge_vdisp,'
header += 'disk_vcirc,disk_smass,disk_size,disk_height,'
header += 'halo_central_smass,halo_break,halo_slope_in,halo_slope_out,halo_q_in,halo_q_out'
# np.savetxt("nehod/data/gse_prob_cond.csv", out, delimiter=',', header=header, comments='')

In [ ]:
len(header.split(",")), out.shape[1]

In [ ]:
arr = np.loadtxt(NEHOD_TORCH + "/example/data/gse_prob_dylan_cond.csv", delimiter=',', skiprows=1)

In [ ]:
arr.shape

In [ ]:
for i in range(13):
    plt.figure()
    plt.hist(arr[:,i])
    plt.show()

In [ ]:
data = arr
tcut = data[:,7] < 0.5
gse_tcut = (data[:,8] < 0.5) & (data[:,8] > 0.33)
mcut = data[:,9] < 5 
fcut = data[:,11] > 0.5
bcut = data[:,12] > 0.5
gse_cut = tcut & gse_tcut & mcut & fcut & bcut

In [ ]:
np.sum(gse_cut) / 1019